# Overfitting and Regularization — Exercises

Companion to the note [Overfitting and Regularization](Overfitting%20and%20Regularization.md).

Practise diagnosing overfitting and the regularizers from the note: ridge (closed form, shrinkage, MAP view), lasso (sparsity, soft-thresholding, coordinate descent), elastic net, early stopping, dropout and data augmentation / noise injection, and choosing $\lambda$ on validation data.

**16 exercises** · 🧠 Concept ×4 · ✍️ By hand ×6 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
rng = np.random.default_rng(0)

## Part A · Concepts

### Exercise 1 · Read the learning curves
*🧠 Concept · ★☆☆*

Diagnose each situation (overfitting, underfitting, fine, or a bug) and suggest one action.

(a) Train loss 0.01, validation loss 0.90 and rising. (b) Train 0.80, validation 0.82, both flat. (c) Train 0.20, validation 0.22.
(d) Validation loss is consistently **lower** than training loss, while dropout is active during training.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Compare the gap (variance) and the level (bias). Remember what dropout does at training vs. evaluation time.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Overfitting** (low train, high and rising val): more regularization, early stopping, more data, smaller model.
(b) **Underfitting** (both high, small gap): larger model, better features, less regularization.
(c) **Fine**: small gap at a low level.
(d) Not necessarily a bug: dropout (and data augmentation) make the *training* loss harder; at evaluation the full network is used. It can also indicate an
easier validation set or leakage; check the split. See [[Bias-Variance Tradeoff]].

</details>

### Exercise 2 · Why does L1 give sparse solutions?
*🧠 Concept · ★★☆*

Using the constrained form $\min\ \text{loss}(w)$ s.t. $\lVert w\rVert_1\le t$ (vs. $\lVert w\rVert_2\le t$), explain geometrically why the lasso solution tends to have exact
zeros while ridge does not. Give a second explanation via the gradient of the penalty near $w_j = 0$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Draw the elliptical loss contours meeting a diamond vs. a disc in 2-D.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The L1 ball is a diamond (cross-polytope) with **corners on the axes**; expanding loss ellipses usually touch it first at a corner, where some $w_j = 0$.
The L2 ball is round, so the touching point is generically off the axes.
Gradient view: the L2 penalty's derivative $2\lambda w_j$ vanishes as $w_j\to0$, so it never pushes a weight exactly to 0. The L1 penalty's (sub)gradient has
constant magnitude $\lambda$, so whenever $|\partial \text{loss}/\partial w_j| \le \lambda$ at $w_j = 0$, staying at exactly 0 is optimal.

</details>

### Exercise 3 · Elastic net and correlated features
*🧠 Concept · ★★☆*

Two features are almost identical copies. What do lasso, ridge and elastic net do with their coefficients? Why does the note describe elastic net as
"sparse but stable"?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

For the lasso, splitting weight $w$ as $(w, 0)$ or $(w/2, w/2)$ gives the same L1 norm. What about the L2 norm?

</details>

<details>
<summary><b>✅ Solution</b></summary>

**Lasso** is indifferent between splits (same $\lVert\cdot\rVert_1$) and typically picks **one** of the two arbitrarily; which one can flip with small data changes (unstable).
**Ridge** prefers the even split $(w/2, w/2)$, which has smaller $\lVert\cdot\rVert_2^2$, but never zeros anything. **Elastic net** inherits the grouping effect of
L2 (correlated features get similar weights) and the sparsity of L1 (irrelevant features are dropped).

</details>

### Exercise 4 · Implicit regularizers
*🧠 Concept · ★☆☆*

For early stopping, dropout, data augmentation, pruning/max-depth and margin maximisation: what does each restrict, and for which model families from the
note is it typically used?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Some regularizers add a penalty term, others limit the optimisation or change the data.

</details>

<details>
<summary><b>✅ Solution</b></summary>

- **Early stopping**: limits how far the optimiser travels from the initialisation (for linear least squares with GD it behaves like ridge); any iteratively trained model, especially [[Neural Networks]] and boosting.
- **Dropout**: randomly zeroes activations, preventing co-adaptation; an implicit ensemble of sub-networks. Neural networks.
- **Data augmentation**: encodes invariances (shifts, flips, noise) by enlarging the training set; images, audio, text.
- **Pruning / max depth**: caps tree complexity ([[Decision Trees]]).
- **Margin maximisation**: among separating hyperplanes prefer the one with smallest $\lVert w\rVert$ ([[Support Vector Machines]]), an L2 penalty in disguise.

</details>

## Part B · By hand

### Exercise 5 · Ridge by hand
*✍️ By hand · ★☆☆*

Compute the ridge solution $w = (X^\top X + \lambda I)^{-1}X^\top y$ (no intercept) for
$X = \begin{pmatrix}1&0\\0&1\\1&1\end{pmatrix}$, $y = (1, 2, 2)^\top$, $\lambda = 1$. Compare with the OLS solution.

In [ ]:
w_ridge_hand = None

_X = np.array([[1,0],[0,1],[1,1.]]); _y = np.array([1,2,2.])
check('ridge', w_ridge_hand, Ridge(alpha=1.0, fit_intercept=False).fit(_X, _y).coef_)

<details>
<summary><b>💡 Hint</b></summary>

$X^\top X = \begin{pmatrix}2&1\\1&2\end{pmatrix}$, $X^\top y = (3, 4)^\top$. The inverse of $\begin{pmatrix}a&b\\b&a\end{pmatrix}$ is $\frac{1}{a^2-b^2}\begin{pmatrix}a&-b\\-b&a\end{pmatrix}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$X^\top X + I = \begin{pmatrix}3&1\\1&3\end{pmatrix}$, inverse $\frac18\begin{pmatrix}3&-1\\-1&3\end{pmatrix}$, so $w = \frac18(9-4,\ -3+12) = (0.625, 1.125)$.
OLS: $\frac13\begin{pmatrix}2&-1\\-1&2\end{pmatrix}(3,4)^\top = (2/3, 5/3)$. Ridge shrinks both, with $\lVert w\rVert$ dropping from 1.80 to 1.29.

```python
w_ridge_hand = [0.625, 1.125]
```

</details>

### Exercise 6 · Ridge shrinks, lasso thresholds
*✍️ By hand · ★★☆*

Assume an orthonormal design, $X^\top X = I$, with OLS solution $\hat w = (3, -0.5, 1.2)$. Show that

- ridge, $\min \tfrac12\lVert y - Xw\rVert^2 + \tfrac\lambda2\lVert w\rVert^2$, gives $w_j = \hat w_j/(1+\lambda)$;
- lasso, $\min \tfrac12\lVert y - Xw\rVert^2 + \lambda\lVert w\rVert_1$, gives the soft-threshold $w_j = \operatorname{sign}(\hat w_j)\max(|\hat w_j| - \lambda, 0)$.

Evaluate both for $\lambda = 1$.

In [ ]:
w_ridge_orth = None
w_lasso_orth = None

# build an explicit orthonormal design and solve with sklearn (sklearn divides the loss by N, hence the scaling of alpha)
_Q = np.linalg.qr(np.random.default_rng(1).normal(size=(10, 3)))[0]
_y = _Q @ np.array([3, -0.5, 1.2]) + (np.eye(10) - _Q @ _Q.T) @ np.random.default_rng(2).normal(size=10)
check('ridge', w_ridge_orth, Ridge(alpha=1.0, fit_intercept=False).fit(_Q, _y).coef_)
check('lasso', w_lasso_orth, Lasso(alpha=1.0 / 10, fit_intercept=False, tol=1e-12, max_iter=100000).fit(_Q, _y).coef_, tol=1e-5)

<details>
<summary><b>💡 Hint</b></summary>

With $X^\top X = I$ the loss separates per coordinate: $\tfrac12\lVert y - Xw\rVert^2 = \tfrac12\lVert w - \hat w\rVert^2 + \text{const}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Per coordinate: ridge minimises $\tfrac12(w_j-\hat w_j)^2 + \tfrac\lambda2 w_j^2 \Rightarrow w_j = \hat w_j/(1+\lambda)$: $(1.5, -0.25, 0.6)$.
Lasso minimises $\tfrac12(w_j-\hat w_j)^2 + \lambda|w_j|$; for $w_j\neq0$ the optimum is $\hat w_j - \lambda\operatorname{sign}(w_j)$, valid only if $|\hat w_j|>\lambda$, otherwise 0:
$(2, 0, 0.2)$. Ridge shrinks **proportionally**, lasso subtracts a **constant** and sets small coefficients exactly to zero.

```python
w_ridge_orth = [1.5, -0.25, 0.6]
w_lasso_orth = [2.0, 0.0, 0.2]
```

</details>

### Exercise 7 · Ridge is MAP with a Gaussian prior
*✍️ By hand · ★★☆*

Assume $y_i = w^\top x_i + \varepsilon_i$, $\varepsilon_i\sim\mathcal N(0,\sigma^2)$, and the prior $w\sim\mathcal N(0, \tau^2 I)$. Show that the MAP estimate minimises
$\lVert y - Xw\rVert^2 + \lambda\lVert w\rVert^2$ and find $\lambda$. Evaluate for $\sigma^2 = 1$, $\tau^2 = 0.25$. Which prior gives the lasso?

In [ ]:
lam_map = None

# numerical MAP: maximise log posterior on random data and compare with ridge(alpha = lam_map)
from scipy.optimize import minimize
_r = np.random.default_rng(3); _X = _r.normal(size=(20, 2)); _y = _X @ [1, -1] + _r.normal(size=20)
_neglogpost = lambda w: np.sum((_y - _X @ w) ** 2) / (2 * 1.0) + np.sum(w ** 2) / (2 * 0.25)
_wmap = minimize(_neglogpost, np.zeros(2)).x
check('MAP = ridge with this lambda', None if lam_map is None else Ridge(alpha=lam_map, fit_intercept=False).fit(_X, _y).coef_, _wmap, tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

$-\ln p(w\mid y) = \frac{1}{2\sigma^2}\lVert y - Xw\rVert^2 + \frac{1}{2\tau^2}\lVert w\rVert^2 + \text{const}$. Multiply by $2\sigma^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Multiplying the negative log posterior by $2\sigma^2$: $\lVert y-Xw\rVert^2 + \frac{\sigma^2}{\tau^2}\lVert w\rVert^2$, so $\lambda = \sigma^2/\tau^2 = 4$.
A narrow prior (small $\tau^2$) means strong regularization. A **Laplace** prior $p(w_j)\propto e^{-|w_j|/b}$ gives the L1 penalty (lasso), as in the note's table.

```python
lam_map = 4.0
```

</details>

### Exercise 8 · Shrinkage factors and effective degrees of freedom
*✍️ By hand · ★★★*

With the SVD $X = UDV^\top$, ridge fitted values are $\hat y = \sum_j u_j \frac{d_j^2}{d_j^2+\lambda} u_j^\top y$.
For singular values $d = (10, 1, 0.1)$ and $\lambda = 1$, compute the three shrinkage factors and the effective degrees of freedom
$\mathrm{df}(\lambda) = \sum_j \frac{d_j^2}{d_j^2+\lambda}$. Which directions are shrunk most?

In [ ]:
shrink = None   # three factors
df_eff = None

_r = np.random.default_rng(4)
_U = np.linalg.qr(_r.normal(size=(8, 3)))[0]; _V = np.linalg.qr(_r.normal(size=(3, 3)))[0]
_X = _U @ np.diag([10, 1, 0.1]) @ _V.T
_H = _X @ np.linalg.solve(_X.T @ _X + np.eye(3), _X.T)       # hat matrix of ridge
check('effective df = trace(H)', df_eff, np.trace(_H))
check('shrinkage factors = eigenvalues of H', None if shrink is None else np.sort(shrink), np.sort(np.linalg.eigvalsh(_H))[-3:], tol=1e-6)

<details>
<summary><b>💡 Hint</b></summary>

$100/101$, $1/2$, $0.01/1.01$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Factors $(0.990, 0.5, 0.0099)$, $\mathrm{df} = 1.5$. Directions with small singular values (low variance in the data, poorly determined coefficients)
are shrunk the most; ridge is a soft version of keeping only the top principal components ([[PCA]]). $\mathrm{df}$ goes from $p=3$ ($\lambda=0$) to 0 ($\lambda\to\infty$).

```python
shrink = [100 / 101, 0.5, 0.01 / 1.01]
df_eff = 100 / 101 + 0.5 + 0.01 / 1.01
```

</details>

### Exercise 9 · Inverted dropout keeps the expectation
*✍️ By hand · ★★☆*

Inverted dropout with keep-probability $p$ outputs $\tilde a = \frac{m}{p}a$ with $m\sim\text{Bernoulli}(p)$. For $a = 2$ and $p = 0.8$, compute $\mathbb E[\tilde a]$ and
$\mathrm{Var}[\tilde a]$. Why is the $1/p$ scaling done at training time, and what happens at test time?

In [ ]:
E_drop = None
Var_drop = None

_m = np.random.default_rng(5).random(2_000_000) < 0.8
_s = _m / 0.8 * 2.0
check('expectation', E_drop, _s.mean(), tol=1e-2); check('variance', Var_drop, _s.var(), tol=1e-2)

<details>
<summary><b>💡 Hint</b></summary>

$\mathbb E[m] = p$, $\mathrm{Var}[m] = p(1-p)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathbb E[\tilde a] = \frac{p}{p}a = 2$. $\mathrm{Var}[\tilde a] = \frac{a^2}{p^2}p(1-p) = a^2\frac{1-p}{p} = 4\cdot0.25 = 1$.
Scaling during training keeps expected activations equal to those at test time, when dropout is switched off and the full network is used unchanged
(an approximation of averaging the ensemble of thinned networks).

```python
E_drop = 2.0
Var_drop = 1.0
```

</details>

### Exercise 10 · When does the lasso switch everything off?
*✍️ By hand · ★★★*

For sklearn's lasso objective $\frac{1}{2N}\lVert y - Xw\rVert^2 + \alpha\lVert w\rVert_1$ (no intercept), show using the subgradient at $w=0$ that $w=0$ is optimal iff
$\alpha \ge \alpha_{\max} = \frac1N\lVert X^\top y\rVert_\infty$. Compute $\alpha_{\max}$ for $X = \begin{pmatrix}1&2\\2&0\\3&1\end{pmatrix}$, $y = (1,1,2)^\top$.

In [ ]:
alpha_max = None

_X = np.array([[1,2],[2,0],[3,1.]]); _y = np.array([1,1,2.])
if alpha_max is not None:
    check('all zero just above alpha_max', np.allclose(Lasso(alpha=alpha_max * 1.001, fit_intercept=False, tol=1e-12).fit(_X, _y).coef_, 0), True)
    check('not all zero just below', np.allclose(Lasso(alpha=alpha_max * 0.95, fit_intercept=False, tol=1e-12).fit(_X, _y).coef_, 0), False)
else:
    check('alpha_max', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

At $w=0$ the gradient of the loss is $-\frac1N X^\top y$; the subdifferential of $\alpha|w_j|$ at 0 is $[-\alpha, \alpha]$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Optimality: $0 \in -\frac1N X^\top y + \alpha\,\partial\lVert w\rVert_1|_{w=0} = -\frac1N X^\top y + [-\alpha,\alpha]^p$, i.e. $|\frac1N x_j^\top y|\le\alpha$ for all $j$.
Here $X^\top y = (9, 4)$, so $\alpha_{\max} = 9/3 = 3$. Regularization paths (e.g. `lasso_path`) start at $\alpha_{\max}$ and decrease on a log grid.

```python
alpha_max = 3.0
```

</details>

## Part C · Code

### Exercise 11 · Ridge path from scratch
*💻 Code · ★☆☆*

Implement `ridge(X, y, lam)` (no intercept, closed form via `np.linalg.solve`) and compute the coefficient path for `lams = np.logspace(-2, 4, 25)` on the data below.
Store it as an array `path` of shape (25, 5). Verify that $\lVert w(\lambda)\rVert_2$ decreases monotonically.

In [ ]:
X_r = rng.normal(size=(40, 5))
y_r = X_r @ np.array([3, -2, 0, 1, 0.5]) + rng.normal(size=40)
lams = np.logspace(-2, 4, 25)

def ridge(X, y, lam):
    return None

path = None

check('path', path, np.array([Ridge(alpha=l, fit_intercept=False).fit(X_r, y_r).coef_ for l in lams]))
check('norm decreases', None if path is None else bool(np.all(np.diff(np.linalg.norm(path, axis=1)) < 0)), True)

<details>
<summary><b>💡 Hint</b></summary>

`np.linalg.solve(X.T @ X + lam * np.eye(X.shape[1]), X.T @ y)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Every coefficient shrinks smoothly toward 0 but none becomes exactly 0. The norm is monotone because, in the SVD basis, each component is multiplied by
$d_j/(d_j^2+\lambda)$, which decreases in $\lambda$.

```python
X_r = rng.normal(size=(40, 5))
y_r = X_r @ np.array([3, -2, 0, 1, 0.5]) + rng.normal(size=40)
lams = np.logspace(-2, 4, 25)

def ridge(X, y, lam):
    return np.linalg.solve(X.T @ X + lam * np.eye(X.shape[1]), X.T @ y)

path = np.array([ridge(X_r, y_r, l) for l in lams])
```

</details>

### Exercise 12 · Lasso by coordinate descent
*💻 Code · ★★★*

Implement `lasso_cd(X, y, alpha, sweeps)` for sklearn's objective $\frac{1}{2N}\lVert y-Xw\rVert^2 + \alpha\lVert w\rVert_1$ (no intercept) by cyclic coordinate descent:
for each $j$, $\rho_j = x_j^\top(y - Xw + x_jw_j)$ and $w_j \leftarrow S(\rho_j, N\alpha)/\lVert x_j\rVert^2$, with the soft-threshold
$S(z,t) = \operatorname{sign}(z)\max(|z|-t, 0)$. Count the non-zeros.

In [ ]:
X_l = rng.normal(size=(100, 20))
w_true = np.r_[np.array([4, -3, 2, 0, 0]), np.zeros(15)]
y_l = X_l @ w_true + rng.normal(size=100)

def soft(z, t):
    return None

def lasso_cd(X, y, alpha, sweeps=500):
    return None

w_cd = lasso_cd(X_l, y_l, alpha=0.2)

check('matches sklearn Lasso', w_cd, Lasso(alpha=0.2, fit_intercept=False, tol=1e-12, max_iter=100000).fit(X_l, y_l).coef_, tol=1e-5)

<details>
<summary><b>💡 Hint 1</b></summary>

Keep a running residual `r = y - X @ w` and update it after each coordinate change: `r -= X[:, j] * (w_new - w_old)`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`np.sign(z) * np.maximum(np.abs(z) - t, 0)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Each coordinate update solves the 1-D lasso exactly (the soft-threshold from the orthonormal exercise). The solution has only a handful of non-zeros: the three true
features plus a couple of tiny spurious ones (here features 8 and 11). This is how sklearn and glmnet solve the lasso.

```python
X_l = rng.normal(size=(100, 20))
w_true = np.r_[np.array([4, -3, 2, 0, 0]), np.zeros(15)]
y_l = X_l @ w_true + rng.normal(size=100)

def soft(z, t):
    return np.sign(z) * np.maximum(np.abs(z) - t, 0)

def lasso_cd(X, y, alpha, sweeps=500):
    N, d = X.shape
    w = np.zeros(d); r = y.copy(); sq = (X ** 2).sum(0)
    for _ in range(sweeps):
        for j in range(d):
            rho = X[:, j] @ r + sq[j] * w[j]
            new = soft(rho, N * alpha) / sq[j]
            r -= X[:, j] * (new - w[j])
            w[j] = new
    return w

w_cd = lasso_cd(X_l, y_l, alpha=0.2)
print('non-zeros:', np.flatnonzero(np.abs(w_cd) > 1e-10), w_cd[:5].round(3))
```

</details>

### Exercise 13 · Taming a degree-15 polynomial
*💻 Code · ★★☆*

Fit degree-15 polynomial features (standardised) to 15 noisy samples of $\sin(2\pi x)$. For $\lambda$ in `np.logspace(-8, 2, 21)` record train and **validation**
MSE (ridge). Store the validation-optimal $\lambda$ in `lam_best`, and the validation MSE at the smallest $\lambda$ and at `lam_best` in `val_tiny`, `val_best`.

In [ ]:
def make(n, seed):
    r = np.random.default_rng(seed); x = np.sort(r.uniform(0, 1, n))
    return x[:, None], np.sin(2 * np.pi * x) + 0.25 * r.normal(size=n)
x_tr, y_tr = make(15, 1); x_va, y_va = make(200, 2)
lam_grid = np.logspace(-8, 2, 21)
lam_best = None
val_tiny = None
val_best = None

def _val(l):
    m = make_pipeline(PolynomialFeatures(15, include_bias=False), StandardScaler(), Ridge(alpha=l)).fit(x_tr, y_tr)
    return np.mean((m.predict(x_va) - y_va) ** 2)
_v = [_val(l) for l in lam_grid]
check('best lambda', lam_best, lam_grid[int(np.argmin(_v))])
check('regularization helps a lot', None if val_best is None else val_best < 0.5 * val_tiny, True)

<details>
<summary><b>💡 Hint</b></summary>

`make_pipeline(PolynomialFeatures(15, include_bias=False), StandardScaler(), Ridge(alpha=l))`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

At $\lambda\approx0$ the polynomial wiggles through the noise (tiny training error, large validation error). A moderate $\lambda$ gives a smooth curve near the noise
floor $0.25^2 = 0.0625$. Training error increases monotonically with $\lambda$; validation error is U-shaped. Pick $\lambda$ on validation (or CV), never on test.

```python
def make(n, seed):
    r = np.random.default_rng(seed); x = np.sort(r.uniform(0, 1, n))
    return x[:, None], np.sin(2 * np.pi * x) + 0.25 * r.normal(size=n)
x_tr, y_tr = make(15, 1); x_va, y_va = make(200, 2)
lam_grid = np.logspace(-8, 2, 21)
tr_mse, va_mse = [], []
for l in lam_grid:
    m = make_pipeline(PolynomialFeatures(15, include_bias=False), StandardScaler(), Ridge(alpha=l)).fit(x_tr, y_tr)
    tr_mse.append(np.mean((m.predict(x_tr) - y_tr) ** 2)); va_mse.append(np.mean((m.predict(x_va) - y_va) ** 2))
lam_best = lam_grid[int(np.argmin(va_mse))]
val_tiny, val_best = va_mse[0], min(va_mse)
print('lambda*', lam_best, '| val MSE tiny/best', round(val_tiny, 3), round(val_best, 3))
```

</details>

### Exercise 14 · Early stopping
*💻 Code · ★★☆*

Train the same degree-15 standardised features (no penalty) by full-batch gradient descent on the MSE with step $1/L$, where $L$ is the largest eigenvalue of the Hessian
$\frac2N\Phi^\top\Phi$. Record train and validation MSE for 20 000 epochs; store the epoch with lowest validation MSE in `best_epoch` and the curves in
`train_curve`, `val_curve`.

In [ ]:
feat = make_pipeline(PolynomialFeatures(15, include_bias=False), StandardScaler()).fit(x_tr)
Phi_tr = np.c_[np.ones(len(x_tr)), feat.transform(x_tr)]
Phi_va = np.c_[np.ones(len(x_va)), feat.transform(x_va)]
best_epoch = None
train_curve = None
val_curve = None

if val_curve is not None:
    check('training loss never increases', bool(np.all(np.diff(train_curve) <= 1e-12)), True)
    check('best epoch is the validation minimum', best_epoch, int(np.argmin(val_curve)))
    check('stopping early beats training to the end', val_curve[best_epoch] < val_curve[-1], True)
else:
    check('early stopping', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`H = 2 / N * Phi.T @ Phi`; `L = np.linalg.eigvalsh(H).max()`; update `w -= (1 / L) * 2 / N * Phi.T @ (Phi @ w - y)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With step $1/L$ GD decreases the training loss monotonically. Starting from $w=0$, GD fits the large-singular-value directions first; small ones (the wiggly, noise-fitting
directions) are learned only late. Stopping at the validation minimum therefore acts like ridge with $\lambda \approx 1/(\eta t)$: *implicit regularization*.

```python
feat = make_pipeline(PolynomialFeatures(15, include_bias=False), StandardScaler()).fit(x_tr)
Phi_tr = np.c_[np.ones(len(x_tr)), feat.transform(x_tr)]
Phi_va = np.c_[np.ones(len(x_va)), feat.transform(x_va)]
N = len(y_tr)
L = np.linalg.eigvalsh(2 / N * Phi_tr.T @ Phi_tr).max()
w = np.zeros(Phi_tr.shape[1]); train_curve, val_curve = [], []
for t in range(20000):
    w -= (1 / L) * 2 / N * Phi_tr.T @ (Phi_tr @ w - y_tr)
    train_curve.append(np.mean((Phi_tr @ w - y_tr) ** 2)); val_curve.append(np.mean((Phi_va @ w - y_va) ** 2))
train_curve, val_curve = np.array(train_curve), np.array(val_curve)
best_epoch = int(np.argmin(val_curve))
print('best epoch', best_epoch, '| val at best', val_curve[best_epoch].round(4), '| val at end', val_curve[-1].round(4))
```

</details>

### Exercise 15 · Lasso vs. elastic net on duplicated features
*💻 Code · ★★☆*

`x1` and `x2` are near-copies, `x3..x10` are noise, and $y = x_1 + x_2 + \varepsilon$. Fit `Lasso(alpha=0.1)` and `ElasticNet(alpha=0.1, l1_ratio=0.5)` and store the
first two coefficients of each in `lasso_12`, `enet_12`. Also count non-zero coefficients among the noise features for both.

In [ ]:
n = 200
x1 = rng.normal(size=n); x2 = x1 + 0.01 * rng.normal(size=n)
X_dup = np.c_[x1, x2, rng.normal(size=(n, 8))]
y_dup = x1 + x2 + 0.5 * rng.normal(size=n)
lasso_12 = None
enet_12 = None

if lasso_12 is not None:
    check('lasso puts (almost) all weight on one copy', min(np.abs(lasso_12)) < 0.1 * max(np.abs(lasso_12)), True)
    check('elastic net splits the weight evenly', abs(enet_12[0] - enet_12[1]) < 0.1 * abs(enet_12[0]), True)
else:
    check('lasso vs enet', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`.coef_[:2]` after fitting each model on `X_dup, y_dup`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The lasso gives ≈ $(1.9, 0)$ or similar: one copy carries all the weight. Elastic net gives two nearly equal coefficients (grouping effect) while still zeroing most
noise features, so it is both sparse and stable.

```python
n = 200
x1 = rng.normal(size=n); x2 = x1 + 0.01 * rng.normal(size=n)
X_dup = np.c_[x1, x2, rng.normal(size=(n, 8))]
y_dup = x1 + x2 + 0.5 * rng.normal(size=n)
las = Lasso(alpha=0.1).fit(X_dup, y_dup); en = ElasticNet(alpha=0.1, l1_ratio=0.5).fit(X_dup, y_dup)
lasso_12, enet_12 = las.coef_[:2], en.coef_[:2]
print('lasso', lasso_12.round(3), 'noise nnz', np.sum(las.coef_[2:] != 0))
print('enet ', enet_12.round(3), 'noise nnz', np.sum(en.coef_[2:] != 0))
```

</details>

### Exercise 16 · Noise injection is ridge in disguise
*💻 Code · ★★★*

Data augmentation by **input noise**: replace each row $x_i$ by $M=200$ noisy copies $x_i + \epsilon$, $\epsilon\sim\mathcal N(0, s^2 I)$ with $s=0.5$, and fit plain least squares
(no intercept). Show that the result approximates ridge with $\lambda = N s^2$ (sklearn's `Ridge(alpha=N*s**2, fit_intercept=False)`), and explain why.

In [ ]:
X_n = rng.normal(size=(50, 3))
y_n = X_n @ np.array([1.0, -2.0, 0.5]) + 0.3 * rng.normal(size=50)
s, M = 0.5, 200
w_aug = None

check('augmented OLS ~ ridge', w_aug, Ridge(alpha=50 * s ** 2, fit_intercept=False).fit(X_n, y_n).coef_, tol=0.03)

<details>
<summary><b>💡 Hint 1</b></summary>

`np.repeat(X_n, M, axis=0) + s * rng.normal(size=(50 * M, 3))`, targets `np.repeat(y_n, M)`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$\mathbb E_\epsilon[(y - (x+\epsilon)^\top w)^2] = (y - x^\top w)^2 + s^2\lVert w\rVert^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Averaging over copies, the loss per original sample becomes $(y_i - x_i^\top w)^2 + s^2\lVert w\rVert^2$; summing over $N$ samples gives
$\lVert y - Xw\rVert^2 + Ns^2\lVert w\rVert^2$, the ridge objective (Bishop, 1995). Augmentation that encodes "small input perturbations should not change the output"
is a regularizer.

```python
X_n = rng.normal(size=(50, 3))
y_n = X_n @ np.array([1.0, -2.0, 0.5]) + 0.3 * rng.normal(size=50)
s, M = 0.5, 200
X_aug = np.repeat(X_n, M, axis=0) + s * rng.normal(size=(50 * M, 3))
w_aug = np.linalg.lstsq(X_aug, np.repeat(y_n, M), rcond=None)[0]
print('augmented', w_aug.round(3), '| ridge', Ridge(alpha=50 * s ** 2, fit_intercept=False).fit(X_n, y_n).coef_.round(3))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Overfitting and Regularization](Overfitting%20and%20Regularization.md).*